### 14일 이내 첫 구매 여부

In [3]:
import pandas as pd

# 1. 데이터 불러오기
users = pd.read_csv('C:/Users/USER/Desktop/data/cleaned/users_EDA.csv', parse_dates=['created_at'])
orders = pd.read_csv('C:/Users/USER/Desktop/data/cleaned/orders_EDA.csv', parse_dates=['created_at'])

# 2. 유저 ID 정리
users = users.rename(columns={'id': 'user_id'})

# 3. 첫 주문일 계산
first_orders = orders.groupby('user_id')['created_at'].min().reset_index()
first_orders.columns = ['user_id', 'first_order_date']

# 4. 유저와 첫 주문 병합
users_merged = pd.merge(users, first_orders, on='user_id', how='left')

# 5. 날짜 타입 다시 확인 및 강제 변환
users_merged['created_at'] = pd.to_datetime(users_merged['created_at'], errors='coerce')
users_merged['first_order_date'] = pd.to_datetime(users_merged['first_order_date'], errors='coerce')

# 6. 가입 후 첫 구매까지 걸린 일수 계산
users_merged['days_to_first_order'] = (users_merged['first_order_date'] - users_merged['created_at']).dt.days

# 7. 라벨 생성 (14일 이내 구매 여부)
users_merged['label'] = users_merged['days_to_first_order'].apply(
    lambda x: 1 if pd.notnull(x) and x <= 14 else 0
)

# 8. 결과 확인
print(users_merged[['user_id', 'created_at', 'first_order_date', 'days_to_first_order', 'label']].head())
print("\n 클래스 분포:\n", users_merged['label'].value_counts(normalize=True))

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_27732\3150470276.py:4: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  users = pd.read_csv('C:/Users/USER/Desktop/data/cleaned/users_EDA.csv', parse_dates=['created_at'])


   user_id                created_at          first_order_date  \
0      457 2022-07-19 13:51:00+00:00 2023-10-08 13:51:00+00:00   
1     6578 2023-11-08 18:49:00+00:00                       NaT   
2    36280 2019-08-24 06:10:00+00:00 2020-05-04 06:10:00+00:00   
3    60193 2020-02-15 11:26:00+00:00 2021-03-21 11:26:00+00:00   
4    64231 2020-03-13 06:45:00+00:00 2020-03-27 06:45:00+00:00   

   days_to_first_order  label  
0                446.0      0  
1                  NaN      0  
2                254.0      0  
3                400.0      0  
4                 14.0      1  

 클래스 분포:
 label
0    0.95164
1    0.04836
Name: proportion, dtype: float64


- 가입 직후 구매 유저는 전체의 약 5% 정도임
- 클래스 불균형이 있기 때문에 모델 학습 시 가중치 등을 고려해야함

### 가입 시점 기준 피처 확인

In [11]:
# 1. 데이터 불러오기
users = pd.read_csv("C:/Users/USER/Desktop/data/cleaned/users_EDA.csv")
users = users.rename(columns={"id": "user_id"})

# 2. created_at -> datetime 형 변환 (timezone 고려)
users["created_at"] = pd.to_datetime(users["created_at"], utc=True, format="mixed")

# 3. 성별 원핫 (결측값 처리 포함)
users["gender"] = users["gender"].fillna("Unknown")
gender_ohe = pd.get_dummies(users["gender"], prefix="gender")

# 4. 유입 채널 원핫 (결측값 처리 포함)
users["traffic_source"] = users["traffic_source"].fillna("unknown")
traffic_ohe = pd.get_dummies(users["traffic_source"], prefix="traffic")

# 5. 시간 관련 피처
users["signup_hour"] = users["created_at"].dt.hour
users["signup_weekday"] = users["created_at"].dt.weekday  # 월=0, 일=6

# 6. 지역 (Top 10만 사용, 나머진 etc 처리)
top_states = users["state"].value_counts().head(10).index
users["state_grouped"] = users["state"].apply(lambda x: x if x in top_states else "etc")
state_ohe = pd.get_dummies(users["state_grouped"], prefix="state")

# 7. 최종 피처 결합
static_features = pd.concat([
    users[["user_id", "age", "signup_hour", "signup_weekday", "is_weekend"]],
    gender_ohe,
    traffic_ohe,
    state_ohe
], axis=1)

# 8. 확인
print(static_features.head())

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_27732\933452338.py:2: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  users = pd.read_csv("C:/Users/USER/Desktop/data/cleaned/users_EDA.csv")


   user_id  age  signup_hour  signup_weekday  is_weekend  gender_F  gender_M  \
0      457   65           13               1       False     False      True   
1     6578   34           18               2       False      True     False   
2    36280   13            6               5        True     False      True   
3    60193   64           11               5        True     False      True   
4    64231   25            6               4       False      True     False   

   traffic_Display  traffic_Email  traffic_Facebook  ...  state_California  \
0            False          False             False  ...             False   
1            False          False             False  ...             False   
2            False           True             False  ...             False   
3            False          False             False  ...             False   
4            False          False             False  ...             False   

   state_England  state_Guangdong  state_Hebei  st

### 14일 행동 요약 피처

In [19]:
# 1. 데이터 불러오기
users = pd.read_csv("C:/Users/USER/Desktop/data/cleaned/users_EDA.csv")
events = pd.read_csv("C:/Users/USER/Desktop/data/cleaned/events_EDA.csv")

# 2. 컬럼명 통일
users = users.rename(columns={"id": "user_id"})
events = events.rename(columns={"id": "event_id"})

# 3. 시간 형식 변환 (올바른 컬럼 사용!)
users["created_at"] = pd.to_datetime(users["created_at"], utc=True, format='mixed')
events["created_at"] = pd.to_datetime(events["created_at"], utc=True, format='mixed')

# 4. 가입일 기준 14일 이내 이벤트 필터링
events_merged = pd.merge(events, users[["user_id", "created_at"]], on="user_id", how="inner", suffixes=('', '_signup'))
events_merged["days_since_signup"] = (events_merged["created_at"] - events_merged["created_at_signup"]).dt.days
events_within_14d = events_merged[events_merged["days_since_signup"] <= 14]

# 5. 결과 확인
print(events_within_14d.head())

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_27732\1449353707.py:2: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  users = pd.read_csv("C:/Users/USER/Desktop/data/cleaned/users_EDA.csv")


     event_id  user_id  sequence_number                            session_id  \
25    1041130  79605.0                4  2963f07e-80dd-4889-b443-51f11992fa08   
44     601109  46004.0                3  2ff3ddd8-0e9c-4574-81a0-da41a8bdbd2b   
55     208984  16107.0                3  a315c404-b733-491d-9d68-d7ca3b398cb3   
71    1000066  76413.0                6  460eb565-afbb-42ca-9982-08b62b7dd80b   
117   1087207  83141.0                4  f934c09a-4b1e-4dca-b717-ceb45d7fc45d   

                          created_at      ip_address      city    state  \
25         2023-06-01 03:11:28+00:00    44.39.197.35    Suzhou  Beijing   
44         2024-01-13 23:02:37+00:00  179.44.216.141   Qingdao  Beijing   
55  2024-01-14 02:35:15.743909+00:00   222.77.205.63  Shanghai  Beijing   
71         2024-01-16 00:31:34+00:00  112.84.180.215    Suzhou  Beijing   
117        2019-03-13 13:04:20+00:00   209.232.80.94      Hami  Beijing   

    postal_code browser traffic_source    uri event_type  \
25

### 이벤트 수 기반 피처 만들기

In [34]:
import pandas as pd

# 1. 파일 로드
events = pd.read_csv("C:/Users/USER/Desktop/data/cleaned/events_EDA.csv")

# 1. 날짜 컬럼 datetime 변환
users["created_at"] = pd.to_datetime(users["created_at"], utc=True, format="mixed")
events["created_at"] = pd.to_datetime(events["created_at"], utc=True, format="mixed")

# 2. 컬럼 정리 (id → user_id, created_at → created_at_signup)
users.rename(columns={"id": "user_id", "created_at": "created_at_signup"}, inplace=True)
events.rename(columns={"id": "event_id"}, inplace=True)

# 3. 유저와 이벤트 merge
events_merged = pd.merge(events, users[["user_id", "created_at_signup"]], on="user_id", how="inner")

# 4. 가입 후 14일 이내 이벤트만 필터링
events_merged["days_since_signup"] = (events_merged["created_at"] - events_merged["created_at_signup"]).dt.days
events_14d = events_merged[events_merged["days_since_signup"].between(0, 14)]

# 5. 이벤트 타입별 수 카운트 (피처 생성)
event_features = (
    events_14d
    .pivot_table(index="user_id", columns="event_type", values="event_id", aggfunc="count", fill_value=0)
    .reset_index()
    .rename_axis(columns=None)
)

# 6. 컬럼 이름 정리 (선택)
event_features.columns = ['user_id'] + [f'event_{col}' for col in event_features.columns if col != 'user_id']

# 결과 확인
print(event_features.head())

   user_id  event_cart  event_department  event_home  event_product  \
0      3.0           1                 1           1              1   
1     13.0           2                 2           2              2   
2     18.0           1                 1           1              1   
3     26.0           1                 1           1              1   
4     45.0           2                 2           2              2   

   event_purchase  
0               1  
1               2  
2               1  
3               1  
4               2  


- 데이터가 대체로 두당 구매 횟수가 많지 않기 때문에 단순히 구매 여부로만 구분하는 것 무리가 있음
- '얼마나 자주 활동했는지'가 구매 가능성 예측에 더 유의미 할 것이라고 판단
- 이벤트 수를 피처로 쓰면, '활동이 적으면 구매 가능성도 낮다'라는 신호를 모델이 인식할 수 있다.

### 필요한 데이터 불러오기

In [42]:
event_counts.to_csv("C:/Users/USER/Desktop/data/cleaned/event_counts_by_user.csv", index=False)

In [44]:
# 이벤트 카운트 데이터
event_counts = pd.read_csv("C:/Users/USER/Desktop/data/cleaned/event_counts_by_user.csv")

# 원본 events 데이터
events = pd.read_csv("C:/Users/USER/Desktop/data/cleaned/events_EDA.csv")

# 유저 데이터
users = pd.read_csv("C:/Users/USER/Desktop/data/cleaned/users_EDA.csv")

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_27732\1207893018.py:8: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  users = pd.read_csv("C:/Users/USER/Desktop/data/cleaned/users_EDA.csv")


### 유저 별 타깃 라벨 만들기

In [47]:
# purchase 이벤트만 필터
purchase_events = events[events["event_type"] == "purchase"]

# purchase가 있었던 user_id 추출
purchase_users = purchase_events["user_id"].unique()

# 타깃: 구매한 유저는 1, 아니면 0
event_counts["target_purchased"] = event_counts["user_id"].isin(purchase_users).astype(int)

### 유저 정보 붙이기

In [50]:
# 필요한 피처만 추출
user_features = users[["id", "age", "gender", "hour", "weekday", "is_weekend"]]
user_features = user_features.rename(columns={"id": "user_id"})

# 병합
df = pd.merge(event_counts, user_features, on="user_id", how="left")

# 원핫 인코딩
df = pd.get_dummies(df, columns=["gender"], drop_first=True)

### 학습 / 테스트 셋 분리 & 모델학습

In [61]:
# 1. users 데이터 불러오기 & id → user_id 컬럼명 변경
users = pd.read_csv("C:/Users/USER/Desktop/data/cleaned/users_EDA.csv")
users = users.rename(columns={"id": "user_id"})

# 2. purchase_counts 예시 생성 (예: user_id별 구매횟수 집계)
# 실제로는 구매 데이터에서 집계한 결과를 불러오거나 생성해야 합니다.
# 예시는 임의 데이터프레임 예시입니다.
purchase_counts = pd.DataFrame({
    "user_id": [3, 13, 18, 26, 45],        # 실제 user_id 목록
    "purchase_count": [1, 2, 1, 1, 2]      # 해당 user_id의 구매 횟수
})

# 3. users와 purchase_counts 병합 (구매 이력 없는 사람은 0으로 처리)
df = users.merge(purchase_counts, on="user_id", how="left")
df["purchase_count"] = df["purchase_count"].fillna(0).astype(int)

# 4. 구매 횟수 범주화 함수 정의 (예: 0,1,2,3 이상 카테고리)
def categorize_purchase(x):
    if x == 0:
        return 0
    elif x == 1:
        return 1
    elif x == 2:
        return 2
    else:
        return 3

df['purchase_category'] = df['purchase_count'].apply(categorize_purchase)

# 5. 결과 확인
print(df[['user_id', 'purchase_count', 'purchase_category']].head())


   user_id  purchase_count  purchase_category
0      457               0                  0
1     6578               0                  0
2    36280               0                  0
3    60193               0                  0
4    64231               0                  0


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_27732\1037687608.py:2: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  users = pd.read_csv("C:/Users/USER/Desktop/data/cleaned/users_EDA.csv")


In [91]:
print(purchase_counts.head())
print(purchase_counts['purchase_count'].describe())

   user_id  purchase_count
0        1               1
1        2               1
2        3               4
3        4               1
4        5               1
count    73043.000000
mean         1.505113
std          0.803125
min          1.000000
25%          1.000000
50%          1.000000
75%          2.000000
max          4.000000
Name: purchase_count, dtype: float64


In [93]:
print(users.columns)
print(purchase_counts.columns)

Index(['id', 'first_name', 'last_name', 'email', 'age', 'gender', 'state',
       'street_address', 'postal_code', 'city', 'country', 'latitude',
       'longitude', 'traffic_source', 'created_at', 'year', 'month', 'day',
       'weekday', 'hour', 'is_weekend', 'purchase_count', 'purchase_category'],
      dtype='object')
Index(['user_id', 'purchase_count'], dtype='object')


In [107]:
print(df.columns.tolist())


['id', 'first_name', 'last_name', 'email', 'age', 'gender', 'state', 'street_address', 'postal_code', 'city', 'country', 'latitude', 'longitude', 'traffic_source', 'created_at', 'year', 'month', 'day', 'weekday', 'hour', 'is_weekend']


In [95]:
print(users['purchase_count'].isna().sum())
print(users['purchase_count'].value_counts())

0
purchase_count
0    100000
Name: count, dtype: int64


In [99]:
import pandas as pd
import numpy as np

# 1. users 데이터 불러오기
users = pd.read_csv("C:/Users/USER/Desktop/data/cleaned/users_EDA.csv")

# 2. purchase_count 컬럼 임의 생성 (예시: 0~5 사이 랜덤 값)
np.random.seed(42)
users['purchase_count'] = np.random.randint(0, 6, size=len(users))

# 3. purchase_count 기준으로 purchase_category 생성
def categorize_purchase(x):
    if x == 0:
        return 0  # 구매 없음
    elif x == 1:
        return 1  # 1회 구매
    elif x == 2:
        return 2  # 2회 구매
    else:
        return 3  # 3회 이상 구매

users['purchase_category'] = users['purchase_count'].apply(categorize_purchase)

print(users[['purchase_count', 'purchase_category']].value_counts())

purchase_count  purchase_category
4               3                    16810
1               1                    16799
3               3                    16776
5               3                    16633
0               0                    16592
2               2                    16390
Name: count, dtype: int64


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_27732\1159187232.py:5: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  users = pd.read_csv("C:/Users/USER/Desktop/data/cleaned/users_EDA.csv")


In [117]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import classification_report
import lightgbm as lgb
from sklearn.utils.class_weight import compute_class_weight

# 1. 데이터 불러오기
users = pd.read_csv("C:/Users/USER/Desktop/data/cleaned/users_EDA.csv")
orders = pd.read_csv("C:/Users/USER/Desktop/data/cleaned/orders_EDA.csv")

# 2. 유저별 구매 횟수 집계
purchase_counts = orders.groupby('user_id').size().reset_index(name='purchase_count')

# 3. users와 purchase_counts 병합 (구매기록 없는 유저는 0으로 채우기)
df = users.merge(purchase_counts, left_on='id', right_on='user_id', how='left')
df['purchase_count'] = df['purchase_count'].fillna(0).astype(int)

# 4. 타깃 카테고리 생성
def categorize_purchase(x):
    if x == 0:
        return 0
    elif x == 1:
        return 1
    elif x == 2:
        return 2
    else:
        return 3

df['purchase_category'] = df['purchase_count'].apply(categorize_purchase)

# 5. 범주형 컬럼
cat_cols = ['gender', 'state', 'traffic_source']

# 6. 원핫인코딩
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
cat_features = ohe.fit_transform(df[cat_cols])
cat_feature_names = ohe.get_feature_names_out(cat_cols)
df_ohe = pd.DataFrame(cat_features, columns=cat_feature_names)

# 7. 숫자형 피처만 선택 (불필요한 컬럼 제외)
drop_cols = ['id', 'first_name', 'last_name', 'email', 'created_at', 'purchase_count'] + cat_cols + ['purchase_category', 'user_id']
num_df = df.drop(columns=drop_cols)

# 8. 학습용 데이터 준비
X = pd.concat([num_df.reset_index(drop=True), df_ohe.reset_index(drop=True)], axis=1)
y = df['purchase_category']

# 9. train/test 분할
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 10. 클래스 가중치 계산
classes = np.unique(y_train)
class_weights = compute_class_weight('balanced', classes=classes, y=y_train)
weights = dict(zip(classes, class_weights))

# 11. LightGBM 데이터셋 변환
train_data = lgb.Dataset(X_train, label=y_train)
valid_data = lgb.Dataset(X_test, label=y_test)

# 12. 모델 파라미터 설정
params = {
    'objective': 'multiclass',
    'num_class': len(classes),
    'metric': 'multi_logloss',
    'learning_rate': 0.05,
    'seed': 42,
    'verbose': -1,
    'class_weight': weights
}

# 13. 모델 학습
model = lgb.train(
    params,
    train_data,
    valid_sets=[valid_data],
    num_boost_round=1000,
    callbacks=[lgb.early_stopping(stopping_rounds=50)]
)

# 14. 예측 및 평가
y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

print(classification_report(y_test, y_pred))

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_27732\3513854289.py:8: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  users = pd.read_csv("C:/Users/USER/Desktop/data/cleaned/users_EDA.csv")


ValueError: pandas dtypes must be int, float or bool.
Fields with bad pandas dtypes: street_address: object, postal_code: object, city: object, country: object